# Assignment 05 · Notebook 06: So sánh và đánh giá

**Sinh viên:** Nguyễn Duy Nghĩa · B23DCCN600 · D23CTPM01 · **GVHD:** PGS.TS Trần Đình Quế

Notebook phục vụ **mục 5** của đề. Nó không huấn luyện gì thêm: đọc số liệu mà notebook 03, 04, 05
đã ghi, nạp lại checkpoint để lấy dự đoán trên tập test, rồi xuất dữ liệu cho các hình so sánh.

Tiêu chí so sánh gồm năm nhóm: **chất lượng** (accuracy, macro-F1, top-5, recall, ROC-AUC),
**chi phí** (tham số, MACs, thời gian/epoch, thời gian suy luận), **hội tụ** (đường học, epoch tốt
nhất), **lỗi** (ma trận nhầm lẫn) và **biểu diễn** (bộ lọc lớp 1, feature map).

In [1]:
import sys
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import torch
from a05 import metrics as M
from a05.data import GPUImageData, load_cifar
from a05.export import MODELS, load_metrics, save_metrics, write_dat, write_image_dat, write_matrix_dat
from a05.models import MODEL_LABELS, MODEL_NAMES, build_model, first_conv
from a05.train import IMAGE_CFG, predict

device = torch.device("cuda")
print("Thiết bị:", torch.cuda.get_device_name(device))
res = {ds: load_metrics(ds) for ds in ("cifar10", "cifar100", "diabetes")}

Thiết bị: NVIDIA GeForce RTX 4060 Laptop GPU


## 1. Bảng tổng hợp 12 lượt chạy (3 tập × 4 mô hình)

In [2]:
rows = []
for ds, r in res.items():
    for name in MODEL_NAMES:
        m = r["models"][name]
        rows.append({"tập": ds, "mô hình": MODEL_LABELS[name], "acc": 100 * m["acc"],
                     "F1": 100 * (m["macro_f1"] if "macro_f1" in m else m["f1"]),
                     "tham số": m["params"], "MACs": m["macs"], "s/epoch": m["epoch_s"],
                     "ms/mẫu": m["infer_ms"], "epoch tốt nhất": m["best_epoch"]})
summary = pd.DataFrame(rows)
summary.round(3)

,tập,mô hình,acc,F1,tham số,MACs,s/epoch,ms/mẫu,epoch tốt nhất
0,cifar10,BasicCNN,85.620,85.604,620362,10848768,1.998,0.005,30
1,cifar10,VGGNet,92.650,92.649,2698954,379259392,50.077,0.099,30
2,cifar10,ResNet,92.690,92.685,2741002,383650304,62.396,0.122,30
3,cifar10,SE-ResNet,92.050,92.032,2763458,383671808,63.427,0.115,28
4,cifar100,BasicCNN,57.480,57.426,643492,10871808,2.660,0.008,30
5,cifar100,VGGNet,70.710,70.786,2722084,379282432,49.179,0.093,29
6,cifar100,ResNet,71.710,71.709,2764132,383673344,58.276,0.106,29
7,cifar100,SE-ResNet,71.860,71.917,2786588,383694848,59.434,0.115,29
8,diabetes,BasicCNN,96.748,79.776,12065,34064,0.596,0.001,22
9,diabetes,VGGNet,96.873,80.141,57073,239632,1.940,0.004,6


In [3]:
for ds, r in res.items():
    mm = r["models"]
    cols = {"idx": list(range(4)), "model": [MODEL_LABELS[n].replace("-", "") for n in MODEL_NAMES],
            "short": [f"M{i}" for i in range(4)],
            "acc": [100 * mm[n]["acc"] for n in MODEL_NAMES],
            "params": [mm[n]["params"] / 1e6 for n in MODEL_NAMES],
            "macs": [mm[n]["macs"] / 1e6 for n in MODEL_NAMES],
            "epoch_s": [mm[n]["epoch_s"] for n in MODEL_NAMES],
            "infer_ms": [mm[n]["infer_ms"] for n in MODEL_NAMES]}
    if ds == "diabetes":
        for k in ("f1", "recall", "precision", "roc_auc", "pr_auc"):
            cols[k] = [100 * mm[n][k] for n in MODEL_NAMES]
    else:
        cols["f1"] = [100 * mm[n]["macro_f1"] for n in MODEL_NAMES]
        cols["top5"] = [100 * mm[n]["top5"] for n in MODEL_NAMES]
    write_dat(f"summary_{ds}", cols)

## 2. Mỗi cơ chế mang lại bao nhiêu
Chênh lệch giữa hai mô hình liền kề chỉ khác nhau đúng một cơ chế. Vì chỉ chạy một seed,
chênh lệch dưới 0,5 điểm được coi là nằm trong mức dao động của một lần chạy.

In [4]:
steps = [("basic", "vgg"), ("vgg", "resnet"), ("resnet", "seresnet")]
delta = {}
for ds, r in res.items():
    mm = r["models"]
    key = "acc" if ds != "diabetes" else "f1"
    delta[ds] = {f"{a}_{b}": 100 * (mm[b][key] - mm[a][key]) for a, b in steps}
    delta[ds].update({f"{a}_{b}_params": mm[b]["params"] - mm[a]["params"] for a, b in steps})
    best = max(MODEL_NAMES, key=lambda n: mm[n][key])
    delta[ds]["best"] = MODEL_LABELS[best]
    delta[ds]["best_key"] = best
    print(ds, {k: (round(v, 2) if isinstance(v, float) else v) for k, v in delta[ds].items()})

cifar10 {'basic_vgg': 7.03, 'vgg_resnet': 0.04, 'resnet_seresnet': -0.64, 'basic_vgg_params': 2078592, 'vgg_resnet_params': 42048, 'resnet_seresnet_params': 22456, 'best': 'ResNet', 'best_key': 'resnet'}
cifar100 {'basic_vgg': 13.23, 'vgg_resnet': 1.0, 'resnet_seresnet': 0.15, 'basic_vgg_params': 2078592, 'vgg_resnet_params': 42048, 'resnet_seresnet_params': 22456, 'best': 'SE-ResNet', 'best_key': 'seresnet'}
diabetes {'basic_vgg': 0.37, 'vgg_resnet': 0.0, 'resnet_seresnet': 1.14, 'basic_vgg_params': 45008, 'vgg_resnet_params': 2800, 'resnet_seresnet_params': 2040, 'best': 'SE-ResNet', 'best_key': 'seresnet'}


## 2b. Hội tụ: accuracy train và val ở epoch cuối, khoảng cách giữa hai đường

In [5]:
final = {}
for ds in res:
    final[ds] = {}
    for name in MODEL_NAMES:
        h = pd.read_csv(f"../outputs/figdata/{ds}_{name}_history.dat", sep=" ")
        last = h.iloc[-1]
        final[ds][name] = {"train_acc": last["train_acc"], "val_acc": last["val_acc"],
                           "gap": last["train_acc"] - last["val_acc"], "max_val_acc": h["val_acc"].max(),
                           "argmax_val_acc": int(h.loc[h["val_acc"].idxmax(), "epoch"])}
print(pd.DataFrame({(ds, n): v for ds, d in final.items() for n, v in d.items()}).T.round(2))

                   train_acc  val_acc    gap  max_val_acc  argmax_val_acc
cifar10  basic         90.74    86.52   4.22        86.56            27.0
         vgg           99.42    92.70   6.72        92.70            30.0
         resnet        99.58    92.90   6.68        92.90            30.0
         seresnet      99.48    93.10   6.38        93.10            30.0
cifar100 basic         71.48    56.92  14.56        56.96            28.0
         vgg           96.19    69.82  26.37        70.02            29.0
         resnet        97.08    71.12  25.96        71.12            30.0
         seresnet      96.44    70.82  25.62        70.88            28.0
diabetes basic         89.70    89.00   0.70        91.25            18.0
         vgg           90.17    89.52   0.65        89.88             2.0
         resnet        89.88    89.19   0.69        89.57             8.0
         seresnet      90.03    87.96   2.07        91.07            10.0


## 3. Ma trận nhầm lẫn
CIFAR-10: 10×10 của mô hình tốt nhất. CIFAR-100: gộp 100 lớp mịn thành 20 siêu lớp.
Mỗi hàng chuẩn hoá thành phần trăm, nên đường chéo là recall của từng lớp.

In [6]:
def load_model(ds, name, n_classes):
    m = build_model(name, 2, (3, 32, 32), n_classes).to(device)
    m.load_state_dict(torch.load(MODELS / f"{ds}_{name}.pt", map_location=device))
    return m.eval()


conf = {}
for ds, n_cls in (("cifar10", 10), ("cifar100", 100)):
    data = GPUImageData(ds, device)
    best = delta[ds]["best_key"]
    logits, y = predict(load_model(ds, best, n_cls), data.batches("test", 1000), amp=True)
    pred = logits.argmax(1)
    if ds == "cifar10":
        cm = M.grouped_confusion(y, pred, np.arange(10), 10)
        names = data.class_names
    else:
        coarse = np.zeros(100, dtype=int)
        coarse[data.raw["y_test"]] = data.raw["yc_test"]
        cm = M.grouped_confusion(y, pred, coarse, 20)
        names = data.raw["coarse_names"]
    write_matrix_dat(f"confusion_{ds}", 100 * cm)
    off = cm.copy()
    np.fill_diagonal(off, 0)
    i, j = np.unravel_index(off.argmax(), off.shape)
    conf[ds] = {"model": MODEL_LABELS[best], "diag_min": 100 * cm.diagonal().min(),
                "diag_max": 100 * cm.diagonal().max(), "worst": names[int(cm.diagonal().argmin())],
                "best": names[int(cm.diagonal().argmax())], "top_pair": [names[i], names[j]],
                "top_pair_pct": 100 * off[i, j]}
    if ds == "cifar100":
        conf[ds]["coarse_acc"] = float((coarse[y] == coarse[pred]).mean())
    print(ds, conf[ds])
    del data
    torch.cuda.empty_cache()

cifar10 {'model': 'ResNet', 'diag_min': np.float64(84.1), 'diag_max': np.float64(96.89999999999999), 'worst': 'cat', 'best': 'automobile', 'top_pair': ['cat', 'dog'], 'top_pair_pct': np.float64(7.5)}


cifar100 {'model': 'SE-ResNet', 'diag_min': np.float64(69.0), 'diag_max': np.float64(92.60000000000001), 'worst': 'reptiles', 'best': 'trees', 'top_pair': ['vehicles_1', 'vehicles_2'], 'top_pair_pct': np.float64(7.3999999999999995), 'coarse_acc': 0.8212}


## 4. Bộ lọc lớp 1 và feature map
16 bộ lọc 3×3×3 đầu tiên của mỗi mô hình CIFAR-10, chuẩn hoá min–max từng bộ lọc về $[0, 255]$
để hiển thị như một ảnh RGB nhỏ. Feature map là đầu ra của tầng conv đầu tiên cho cùng một ảnh test.

In [7]:
c10 = load_cifar("cifar10")
data = GPUImageData("cifar10", device)
probe_idx = int(np.flatnonzero(c10["y_test"] == c10["class_names"].index("horse"))[0])
write_image_dat("probe_image", c10["x_test"][probe_idx])
xprobe = data.normalize(data.x["test"][probe_idx:probe_idx + 1])
fstats = {}
for name in MODEL_NAMES:
    model = load_model("cifar10", name, 10)
    conv = first_conv(model)
    W = conv.weight.detach().float().cpu().numpy()[:16]            # (16, 3, 3, 3)
    lo = W.min(axis=(1, 2, 3), keepdims=True)
    hi = W.max(axis=(1, 2, 3), keepdims=True)
    imgs = ((W - lo) / (hi - lo) * 255).round().astype(np.uint8).transpose(0, 2, 3, 1)
    grid = np.full((4 * 3 + 3, 4 * 3 + 3, 3), 255, np.uint8)
    for k in range(16):
        r, c = divmod(k, 4)
        grid[r * 4:r * 4 + 3, c * 4:c * 4 + 3] = imgs[k]
    write_image_dat(f"filters_{name}", grid)
    with torch.no_grad():
        fm = conv(xprobe)[0].float().cpu().numpy()                 # (C, 32, 32)
    for k in range(8):
        f = fm[k]
        write_matrix_dat(f"fmap_{name}_{k}", (f - f.min()) / (f.max() - f.min() + 1e-12))
    fstats[name] = {"n_filters": int(conv.weight.shape[0]), "weight_std": float(W.std())}
print(fstats)

{'basic': {'n_filters': 32, 'weight_std': 0.31888848543167114}, 'vgg': {'n_filters': 64, 'weight_std': 0.15156181156635284}, 'resnet': {'n_filters': 64, 'weight_std': 0.13254210352897644}, 'seresnet': {'n_filters': 64, 'weight_std': 0.10761967301368713}}


## 5. Dò shape tầng theo tầng
Chạy một tensor 0 qua từng tầng con của mỗi mô hình, ghi shape đầu ra và số tham số của tầng đó.

In [8]:
n_feat = load_metrics("data")["diabetes"]["n_features_encoded"]
trace = {}
for name in MODEL_NAMES:
    for tag, dim, shape, n_out in (("2d", 2, (3, 32, 32), 10), ("1d", 1, (1, n_feat), 1)):
        m = build_model(name, dim, shape, n_out).eval()
        x = torch.zeros(1, *shape)
        rows = [["Input", "×".join(map(str, shape)), 0]]
        with torch.no_grad():
            for layer in list(m.features) + list(m.head):
                x = layer(x)
                rows.append([type(layer).__name__, "×".join(map(str, x.shape[1:])),
                             sum(q.numel() for q in layer.parameters())])
        trace[f"{name}_{tag}"] = rows
for r in trace["basic_2d"]:
    print(r)

['Input', '3×32×32', 0]
['Conv2d', '32×32×32', 896]
['ReLU', '32×32×32', 0]
['MaxPool2d', '32×16×16', 0]
['Conv2d', '64×16×16', 18496]
['ReLU', '64×16×16', 0]
['MaxPool2d', '64×8×8', 0]
['Conv2d', '128×8×8', 73856]
['ReLU', '128×8×8', 0]
['MaxPool2d', '128×4×4', 0]
['Flatten', '2048', 0]
['Linear', '256', 524544]
['ReLU', '256', 0]
['Linear', '10', 2570]


In [9]:
import platform
import sklearn
env = {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__,
       "pandas": pd.__version__, "sklearn": sklearn.__version__, "cuda": torch.version.cuda,
       "cudnn": torch.backends.cudnn.version(), "gpu": torch.cuda.get_device_name(device),
       "os": platform.platform(terse=True)}
print(env)
# OneCycleLR mặc định dành 30% số bước đầu để tăng lr lên đỉnh.
lr_peak_epoch = int(round(0.3 * IMAGE_CFG["epochs"]))
_ = save_metrics("compare", {"delta": delta, "confusion": conf, "filters": fstats, "trace": trace,
                             "lr_peak_epoch": lr_peak_epoch, "env": env, "final": final,
                             "probe_class": "horse", "probe_index": probe_idx,
                             "noclip_diverged_epoch": int(np.argmax(np.isnan(np.loadtxt(
                                 "../outputs/figdata/cifar10_basic_noclip_history.dat", skiprows=1)[:, 1]))) + 1})

{'python': '3.13.5', 'torch': '2.13.0+cu126', 'numpy': '2.5.1', 'pandas': '3.0.5', 'sklearn': '1.9.0', 'cuda': '12.6', 'cudnn': 91002, 'gpu': 'NVIDIA GeForce RTX 4060 Laptop GPU', 'os': 'Windows-11'}
